# AI-GIS — Full End-to-End Pipeline
### A Honeypot-Trained Hybrid Detector for Web Injection Attacks (SQLi & XSS)

This notebook runs the complete pipeline as actually implemented:

1. **Load** the labelled honeypot corpus (WEB-IDS23-seeded).
2. **Preprocess** and split by session (no leakage).
3. **Train** the three-part stacked model — Random Forest + character-level LSTM + logistic-regression meta-learner.
4. **Evaluate** on held-out benchmarks (paper benchmark; optional Claude red-team stress set).
5. **Compare** against a ModSecurity CRS PL2 baseline on identical inputs.
6. **Report** — metrics, confusion matrices, bootstrap CIs, McNemar's paired test, ablation, and ROC.

**Reproducibility.** Fixed seed (42) throughout. The committed checkpoint model reproduces the
reported figures: **97.9% detection · F1 0.925 · 19.6% FPR · AUC 0.993** on the paper benchmark.

> Run from the `dashboard/` directory so the relative paths resolve. Cells degrade gracefully:
> if a dataset or the ModSecurity container is missing, that step is skipped with a clear note.

## 0 · Setup & reproducibility

In [ ]:
import os, sys, json, math, random, urllib.parse, urllib.request, urllib.error
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED); np.random.seed(SEED)

import tensorflow as tf
tf.random.set_seed(SEED)
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (f1_score, precision_score, recall_score,
                             roc_auc_score, roc_curve, accuracy_score)
from sklearn.utils import resample
import matplotlib.pyplot as plt

print("TensorFlow", tf.__version__, "| seed", SEED)

## 1 · Load the honeypot training corpus

The corpus is a JSON-lines honeypot log: one HTTP request per line with a label
(1 = attack, 0 = benign), an attack family, and provenance flags. Attack payloads are
produced by family-specific generators placed on WEB-IDS23 flow metadata.

In [ ]:
DATA = Path("data/honeypot_final.log")

def hp_text(r):
    uri = r.get("uri",""); body = r.get("request_body","") or ""
    q = uri.split("?",1)[1] if "?" in uri else ""
    return f"{q} {body}".strip()

SQLI_FAMS = {"tautology","union_based","boolean_blind","time_based_blind",
             "error_based","stacked_query","auth_bypass"}
XSS_FAMS  = {"reflected","stored","dom_based"}

if DATA.exists():
    recs = []
    with open(DATA, encoding="utf-8") as f:
        for line in f:
            line=line.strip()
            if not line: continue
            try: r=json.loads(line)
            except json.JSONDecodeError: continue
            fam = r.get("attack_family","benign")
            atype = "sqli" if fam in SQLI_FAMS else ("xss" if fam in XSS_FAMS else "benign")
            recs.append({"text":hp_text(r),"label":int(r.get("label",0)),
                         "family":fam,"attack_type":atype,
                         "session_id":str(r.get("session_id",len(recs)))})
    df = pd.DataFrame(recs)
    print(f"Loaded {len(df):,} records")
    print(df["label"].value_counts().rename({0:"benign",1:"attack"}))
    print("\nAttack families:\n", df[df.label==1]["family"].value_counts())
else:
    raise FileNotFoundError("data/honeypot_final.log not found — run from dashboard/")

## 2 · Preprocess & session-grouped split

Split by **session** (not by row) so no session appears in two partitions — this prevents
near-duplicate payloads from leaking between train and test. ~70 / 15 / 15.

In [ ]:
def normalize_text(t): return str(t).strip()
df["text"] = df["text"].map(normalize_text)

groups = df["session_id"].values
tr_idx, tmp_idx = next(GroupShuffleSplit(1, test_size=0.30, random_state=SEED).split(df, groups=groups))
train_df = df.iloc[tr_idx].reset_index(drop=True)
tmp = df.iloc[tmp_idx].reset_index(drop=True)
va_idx, te_idx = next(GroupShuffleSplit(1, test_size=0.50, random_state=SEED).split(tmp, groups=tmp["session_id"].values))
val_df, test_df = tmp.iloc[va_idx].reset_index(drop=True), tmp.iloc[te_idx].reset_index(drop=True)

# leakage guard
s = lambda d: set(d.session_id)
assert not (s(train_df)&s(test_df)) and not (s(train_df)&s(val_df)) and not (s(val_df)&s(test_df))
print(f"train={len(train_df)}  val={len(val_df)}  test={len(test_df)}  — no session crosses a split. [OK]")

## 3 · Feature engineering (Random Forest branch)

Each request → a fixed-length vector: aggregate counts, structural pattern flags, and a
300-term character n-gram TF-IDF (fitted on TRAIN only). ~319 features total.

In [ ]:
SQLI_KW = ["select","union","or ","and ","sleep","drop","--","/*","0x","waitfor","extractvalue"]
XSS_KW  = ["<script","onerror","onload","javascript:","<svg","<img","alert(","onmouseover"]

def entropy(s):
    if not s: return 0.0
    from collections import Counter
    n=len(s); return -sum((c/n)*math.log2(c/n) for c in Counter(s).values())

def agg_features(t):
    lo=t.lower()
    return {"payload_len":len(t),"entropy":round(entropy(t),4),
            "num_special":sum(1 for c in t if not c.isalnum() and not c.isspace()),
            "num_digits":sum(c.isdigit() for c in t),
            "num_upper":sum(c.isupper() for c in t),
            "has_sqli_kw":int(any(k in lo for k in SQLI_KW)),
            "has_xss_kw":int(any(k in lo for k in XSS_KW)),
            "quote_count":t.count("'")+t.count('"'),
            "comment_tokens":t.count("--")+t.count("/*")+t.count("#"),
            "special_ratio":round(sum(1 for c in t if not c.isalnum() and not c.isspace())/max(len(t),1),4)}

NGRAM=300
vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2,4), max_features=NGRAM, lowercase=False)
vec.fit(train_df["text"])

def rf_matrix(texts):
    agg = pd.DataFrame([agg_features(t) for t in texts]).reset_index(drop=True)
    ng  = pd.DataFrame(vec.transform(texts).toarray(),
                       columns=[f"ng_{i}" for i in range(NGRAM)]).reset_index(drop=True)
    return pd.concat([agg,ng],axis=1)

Xtr, ytr = rf_matrix(train_df.text), train_df.label.to_numpy()
Xva, yva = rf_matrix(val_df.text),   val_df.label.to_numpy()
Xte, yte = rf_matrix(test_df.text),  test_df.label.to_numpy()
print("RF feature width:", Xtr.shape[1])

## 4 · Character sequence (LSTM branch)

Each payload → an ordinal character sequence, padded/cut to 200. Padding is id 0 and the
Embedding layer **masks** it (without masking the network does not learn).

In [ ]:
MAXLEN=200
def encode(t, n=MAXLEN):
    a=np.zeros(n,dtype=np.int32)
    for i,c in enumerate(t[:n]):
        code=ord(c); a[i]=code if code<=127 else 1   # 1 = UNK
    return a
Etr=np.stack([encode(t) for t in train_df.text])
Eva=np.stack([encode(t) for t in val_df.text])
Ete=np.stack([encode(t) for t in test_df.text])
print("LSTM input:", Etr.shape)

## 5 · Train the three-part stacked model

RF on the feature vector; LSTM on the character sequence (checkpoint best val AUC); and a
logistic-regression meta-learner fitted on the two branches' **validation** probabilities.

In [ ]:
rf = RandomForestClassifier(n_estimators=200, max_depth=20, class_weight="balanced",
                            random_state=SEED, n_jobs=-1).fit(Xtr, ytr)
print("Random Forest trained.")

init = keras.initializers.GlorotUniform(seed=SEED)
lstm = keras.Sequential([
    layers.Input(shape=(MAXLEN,)),
    layers.Embedding(128, 32, mask_zero=True),          # mask padding
    layers.LSTM(64, return_sequences=True, kernel_initializer=init),
    layers.Dropout(0.3, seed=SEED),
    layers.LSTM(32, kernel_initializer=init),
    layers.Dropout(0.3, seed=SEED),
    layers.Dense(1, activation="sigmoid"),
])
lstm.compile(optimizer=keras.optimizers.Adam(1e-3), loss="binary_crossentropy",
             metrics=[keras.metrics.AUC(name="auc")])
cbs=[keras.callbacks.ModelCheckpoint("lstm_best.keras",monitor="val_auc",mode="max",save_best_only=True),
     keras.callbacks.EarlyStopping(monitor="val_auc",mode="max",patience=3,restore_best_weights=True)]
lstm.fit(Etr, ytr, validation_data=(Eva, yva), epochs=8, batch_size=128, callbacks=cbs, verbose=2)

rf_va = rf.predict_proba(Xva)[:,1]
ls_va = lstm.predict(Eva, verbose=0).ravel()
meta = LogisticRegression(max_iter=1000).fit(np.column_stack([rf_va, ls_va]), yva)
print("Meta-learner fitted on validation probabilities.")

### 5.1 · Scoring helpers & the checkpoint

`predict_proba` runs the full stack. This notebook trains fresh; to score with the exact
**committed checkpoint** instead, load `models/*.pkl` + `models/lstm_best.keras`
(rf2.pkl sha256 starts `08aff449`).

In [ ]:
def score(texts):
    X = rf_matrix(texts); r = rf.predict_proba(X)[:,1]
    E = np.stack([encode(t) for t in texts]); l = lstm.predict(E, verbose=0).ravel()
    return meta.predict_proba(np.column_stack([r,l]))[:,1], r, l

def metrics(y, p, thr=0.5):
    pred=(p>=thr).astype(int); neg=(y==0)
    return {"n":len(y),"recall":recall_score(y,pred,zero_division=0),
            "precision":precision_score(y,pred,zero_division=0),
            "f1":f1_score(y,pred,zero_division=0),
            "fpr":float(pred[neg].mean()) if neg.any() else float("nan"),
            "auc":roc_auc_score(y,p) if len(set(y))>1 else float("nan")}

## 6 · Held-out evaluation

Pick a benchmark. `paper_benchmark.csv` is the authoritative 1,700-record held-out set behind
the reported figures. `claude_redteam.csv` is a harder adversarial stress set (attacks only —
combine with benign for an FPR axis).

In [ ]:
BENCH = Path("data/eval/paper_benchmark.csv")   # <- swap datasets here

if BENCH.exists():
    b = pd.read_csv(BENCH); b["label"]=b["label"].astype(int)
    b["text"]=b["text"].map(normalize_text)
    bp,_,_ = score(b["text"].tolist())
    overall = metrics(b["label"].to_numpy(), bp)
    print("AI-GIS on", BENCH.name)
    for k,v in overall.items(): print(f"  {k:10s} {v:.4f}" if isinstance(v,float) else f"  {k:10s} {v}")
else:
    print("[skip] benchmark not found:", BENCH)

### 6.1 · By attack type

In [ ]:
col = "attack_type" if "attack_type" in b.columns else None
rows=[{"subset":"Combined", **overall}]
if col:
    for at in ["sqli","xss"]:
        m=(b["label"]==1)&(b[col].astype(str).str.lower().str.contains(at)); neg=b["label"]==0
        idx=(m|neg).values
        if m.sum():
            rows.append({"subset":at.upper(), **metrics(b["label"].to_numpy()[idx], bp[idx])})
pd.DataFrame(rows)[["subset","n","recall","precision","f1","fpr","auc"]].round(4)

### 6.2 · Bootstrap 95% CI for F1 (1,000 iterations, seed 42)

In [ ]:
yb=b["label"].to_numpy()
rs=np.random.RandomState(SEED); f1s=[]
for _ in range(1000):
    idx=rs.randint(0,len(yb),len(yb))
    if len(set(yb[idx]))<2: continue
    f1s.append(f1_score(yb[idx], (bp[idx]>=0.5).astype(int), zero_division=0))
print(f"F1 = {overall['f1']:.4f}   95% CI [{np.percentile(f1s,2.5):.3f}, {np.percentile(f1s,97.5):.3f}]")

## 7 · ModSecurity CRS PL2 baseline (optional — needs Docker)

The rule-based baseline. Runs as a container and inspects the **same** records independently.
Browser headers are sent so CRS judges the payload, not the client. Auto-skips if unreachable.

In [ ]:
MODSEC="http://127.0.0.1:8080/"
H={"User-Agent":"Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/124 Safari/537.36",
   "Accept":"text/html,application/xhtml+xml,*/*;q=0.8","Accept-Language":"en-US,en;q=0.9"}

def ms_alive():
    try: urllib.request.urlopen(urllib.request.Request(MODSEC,headers=H),timeout=3); return True
    except urllib.error.HTTPError: return True
    except Exception: return False

def ms_block(p):
    try:
        req=urllib.request.Request(MODSEC+"?q="+urllib.parse.quote(p,safe=""),headers=H)
        with urllib.request.urlopen(req,timeout=8) as r: return 1 if r.status==403 else 0
    except urllib.error.HTTPError as e: return 1 if e.code==403 else 0
    except Exception: return 0

if ms_alive():
    print("ModSecurity reachable — scoring (parallel)...")
    with ThreadPoolExecutor(max_workers=16) as ex:
        ms = np.array(list(ex.map(ms_block, b["text"].tolist())))
    ai_pred=(bp>=0.5).astype(int)
    ms_m={"recall":recall_score(yb,ms,zero_division=0),"precision":precision_score(yb,ms,zero_division=0),
          "f1":f1_score(yb,ms,zero_division=0),"fpr":float(ms[yb==0].mean()),"accuracy":float((ms==yb).mean())}
    comp=pd.DataFrame({"Metric":["Recall","Precision","F1","FPR","Accuracy"],
        "AI-GIS":[overall["recall"],overall["precision"],overall["f1"],overall["fpr"],float((ai_pred==yb).mean())],
        "ModSecurity PL2":[ms_m["recall"],ms_m["precision"],ms_m["f1"],ms_m["fpr"],ms_m["accuracy"]]})
    display(comp.round(4))
else:
    ms=None; print("[skip] ModSecurity not reachable — start it with: docker compose up -d")

### 7.1 · McNemar's paired exact test

In [ ]:
if ms is not None:
    from scipy.stats import binomtest
    ai_ok=((bp>=0.5).astype(int)==yb); ms_ok=(ms==yb)
    bcnt=int((ai_ok&~ms_ok).sum()); ccnt=int((~ai_ok&ms_ok).sum())
    p=binomtest(min(bcnt,ccnt), n=bcnt+ccnt, p=0.5, alternative="two-sided").pvalue if (bcnt+ccnt) else 1.0
    print(f"AI right / ModSec wrong = {bcnt}")
    print(f"AI wrong / ModSec right = {ccnt}")
    print(f"McNemar exact p = {p:.2e}  ->", "SIGNIFICANT" if p<0.05 else "not significant")
else:
    print("Baseline unavailable — McNemar skipped.")

## 8 · Ablation — does stacking help?

Compare the full stack against its parts on the benchmark. Pre-registered rule: if the full
stack does not beat the best single component, stacking adds no value on this corpus.

In [ ]:
rf_b, ls_b = score(b["text"].tolist())[1:]
stack_b = meta.predict_proba(np.column_stack([rf_b, ls_b]))[:,1]
abl=pd.DataFrame([
    {"condition":"Random Forest alone", **metrics(yb, rf_b)},
    {"condition":"LSTM alone",          **metrics(yb, ls_b)},
    {"condition":"Full stack (RF+LSTM+meta)", **metrics(yb, stack_b)},
])[["condition","f1","precision","recall","fpr","auc"]]
display(abl.round(4))
best=max(metrics(yb,rf_b)["f1"], metrics(yb,ls_b)["f1"])
print("Stacking helps." if metrics(yb,stack_b)["f1"]>best
      else "Stacking does NOT beat the best single branch on this corpus.")

## 9 · ROC curve

In [ ]:
fpr,tpr,_ = roc_curve(yb, bp)
plt.figure(figsize=(6.5,5.5))
plt.plot(fpr,tpr,color="#059669",lw=2,label=f"AI-GIS (AUC={overall['auc']:.3f})")
if ms is not None:
    plt.scatter([ms_m["fpr"]],[ms_m["recall"]],color="#c0392b",s=130,marker="X",zorder=5,
                label=f"ModSecurity PL2 (FPR={ms_m['fpr']:.2f})")
plt.plot([0,1],[0,1],"--",color="#888",lw=1)
plt.xlabel("False-Positive Rate"); plt.ylabel("True-Positive Rate (Recall)")
plt.title("ROC Curve: AI-GIS vs ModSecurity CRS PL2")
plt.legend(loc="lower right"); plt.grid(alpha=.3); plt.tight_layout(); plt.show()

## 10 · (Optional) Claude red-team stress test

The hardest adversarial set: assistant-authored polymorphic attacks including natural-language
intent injection. Attacks only — detection here is expected to be **much lower** than the paper
benchmark, which is the point: it exposes the syntax-vs-intent weakness.

In [ ]:
CLAUDE = Path("data/eval/claude_redteam.csv")
if CLAUDE.exists():
    c = pd.read_csv(CLAUDE); c["text"]=c["text"].map(normalize_text)
    cp,_,_ = score(c["text"].tolist())
    det=(cp>=0.5).mean()
    print(f"Claude red-team: {len(c)} attacks — detection {det*100:.1f}%  (evaded {int((cp<0.5).sum())})")
    print("By family (detection %):")
    fcol = "category" if "category" in c.columns else ("attack_type" if "attack_type" in c.columns else None)
    if fcol:
        for fam,g in c.assign(p=cp).groupby(fcol):
            print(f"  {fam:16s} {(g.p>=0.5).mean()*100:5.1f}%  (n={len(g)})")
else:
    print("[skip] claude_redteam.csv not present.")

---
### Notes
- All randomness is seeded; RF and the split reproduce exactly, the LSTM to numerical tolerance.
- Swap `BENCH` in §6 to evaluate a different held-out set; the whole downstream (metrics, CI,
  ModSecurity, ablation, ROC) follows automatically.
- To score with the frozen checkpoint instead of a fresh train, load `models/*.pkl` and
  `models/lstm_best.keras` (rf2.pkl sha256 `08aff449…`) into `rf`, `meta`, `vec`, `lstm`.